# Notebook 14 -- Quantum Machine Learning

**Prerequisites:** Notebook 13. Familiarity with variational circuits and optimizers.

Classical machine learning relies on kernel methods, feature spaces, and
gradient-based optimization. Quantum computing offers a new twist on all
three: a quantum computer can prepare exponentially large feature spaces,
compute kernel entries via quantum interference, and train parameterized
circuits with classical optimizers in a hybrid loop.

**Quantum machine learning (QML)** encodes classical data into quantum
states, processes it through parameterized quantum circuits, and extracts
predictions via measurement. The hope is that quantum feature maps can
capture correlations in data that are hard for classical models to learn.

By the end of this notebook you will be able to:

1. **Describe** quantum feature maps and how they encode classical data into quantum states.
2. **Implement** a variational quantum classifier and evaluate its accuracy.
3. **Compute** and interpret a quantum kernel matrix.
4. **Compare** different feature map strategies.

In this notebook we will:

1. Explore **quantum feature maps** -- circuits that encode classical data
   into quantum states (angle embedding, amplitude embedding, Z and ZZ
   feature maps).
2. Build and train a **Variational Quantum Classifier (VQC)** -- a hybrid
   classical-quantum model that classifies data using a parameterized
   quantum circuit.
3. Compute **quantum kernel matrices** -- inner products between quantum
   feature states that can be fed into classical kernel methods.
4. Predict and verify classification on unseen test data.

### Misconception: Quantum ML always outperforms classical ML

Quantum advantage in ML is not guaranteed. For many datasets, classical
models match or exceed quantum models. The potential advantage comes from
quantum feature maps that create classically hard-to-simulate kernel
functions. Whether a given dataset benefits from quantum encoding is an
active research question.

In [1]:
import (
	"context"
	"fmt"
	"math"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/algorithm/ansatz"
	"github.com/splch/goqu/algorithm/optim"
	"github.com/splch/goqu/algorithm/vqc"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/viz"
)

## Quantum Feature Maps

A **feature map** is a circuit that encodes a classical data vector
$\vec{x} \in \mathbb{R}^n$ into a quantum state $|\phi(\vec{x})\rangle$.
Different encoding strategies create different quantum feature spaces:

| Feature Map | Encoding | Expressivity |
|:---|:---|:---|
| **Angle Embedding** | Each feature $x_i$ becomes a rotation angle on qubit $i$ | Linear -- each feature maps to one rotation |
| **Z Feature Map** | H + RZ($x_i$) per qubit, repeated $d$ times | Product state -- no entanglement |
| **ZZ Feature Map** | Like Z, plus RZZ($x_i \cdot x_j$) entangling layers | Captures pairwise feature interactions |
| **Amplitude Embedding** | Feature vector encoded as state amplitudes | Exponentially compact -- $2^n$ features in $n$ qubits |

The choice of feature map determines what patterns the classifier can
learn. More expressive feature maps can capture more complex decision
boundaries, but may also be harder to train.

Let's build circuits with each feature map and inspect their structure.

In [2]:
%%
// Demonstrate different feature maps with sample data
sampleData := []float64{0.3, 0.7}
qubits := []int{0, 1}

// AngleEmbedding: encodes features as rotation angles
angleFM := vqc.AngleEmbedding(vqc.RotRY)

ab := builder.New("angle-ry", 2)
angleFM(ab, sampleData, qubits)
angleCirc, _ := ab.Build()

fmt.Println("=== Angle Embedding (RY) ===")
fmt.Println("Each feature becomes an RY rotation on the corresponding qubit.")
gonbui.DisplayHTML(draw.SVG(angleCirc))

// ZFeatureMap: H + RZ(x) per qubit, repeated depth times
zFM := vqc.ZFeatureMap(2)

zb := builder.New("z-feature", 2)
zFM(zb, sampleData, qubits)
zCirc, _ := zb.Build()

fmt.Println("=== Z Feature Map (depth=2) ===")
fmt.Println("H + RZ(x_i) per qubit, repeated twice. No entanglement.")
gonbui.DisplayHTML(draw.SVG(zCirc))

// ZZFeatureMap: like Z but with RZZ(x_i * x_j) entangling layers
zzFM := vqc.ZZFeatureMap(2)

zzb := builder.New("zz-feature", 2)
zzFM(zzb, sampleData, qubits)
zzCirc, _ := zzb.Build()

fmt.Println("=== ZZ Feature Map (depth=2) ===")
fmt.Println("Like Z, plus RZZ(x_i * x_j) entangling gates between adjacent qubits.")
gonbui.DisplayHTML(draw.SVG(zzCirc))

fmt.Println("The ZZ feature map creates entanglement between qubits, encoding")
fmt.Println("pairwise feature interactions that the Z map cannot capture.")

=== Angle Embedding (RY) ===
Each feature becomes an RY rotation on the corresponding qubit.
=== Z Feature Map (depth=2) ===
H + RZ(x_i) per qubit, repeated twice. No entanglement.
=== ZZ Feature Map (depth=2) ===
Like Z, plus RZZ(x_i * x_j) entangling gates between adjacent qubits.
The ZZ feature map creates entanglement between qubits, encoding
pairwise feature interactions that the Z map cannot capture.


q0 
 
 q1 
 
 RY(0.3) 
 RY(0.7)

q0 
 
 q1 
 
 H 
 RZ(0.3) 
 H 
 RZ(0.7) 
 H 
 RZ(0.3) 
 H 
 RZ(0.7)

q0 
 
 q1 
 
 H 
 RZ(0.3) 
 H 
 RZ(0.7) 
 
 RZZ(0.21) 
 RZZ(0.21) 
 H 
 RZ(0.3) 
 H 
 RZ(0.7) 
 
 RZZ(0.21) 
 RZZ(0.21)

## Amplitude Embedding

While angle embedding uses one qubit per feature, **amplitude embedding**
encodes an entire feature vector into the amplitudes of a quantum state.
An $n$-qubit state has $2^n$ amplitudes, so $n$ qubits can encode up to
$2^n$ classical features -- an exponential compression.

The feature vector $\vec{x}$ is normalized to unit L2 norm and mapped to:

$$|\phi(\vec{x})\rangle = \sum_{i=0}^{2^n - 1} \frac{x_i}{\|\vec{x}\|} |i\rangle$$

This is powerful for high-dimensional data, but the state preparation
circuit can be deep. Let's encode a 4-dimensional feature vector into
2 qubits.

In [3]:
%%
// AmplitudeEmbedding: encode a 4D feature vector into 2 qubits
ampFM := vqc.AmplitudeEmbedding()

features4D := []float64{1.0, 2.0, 3.0, 4.0}
qubits := []int{0, 1}

ampb := builder.New("amp-embed", 2)
ampFM(ampb, features4D, qubits)
ampCirc, _ := ampb.Build()

fmt.Println("=== Amplitude Embedding ===")
fmt.Println("4D feature vector [1, 2, 3, 4] encoded into 2 qubits.")
gonbui.DisplayHTML(draw.SVG(ampCirc))

// Verify the amplitudes match the normalized feature vector
norm := math.Sqrt(1*1 + 2*2 + 3*3 + 4*4)
fmt.Println("Expected amplitudes (normalized):")
for i, f := range features4D {
	fmt.Printf("  |%02b> = %.4f\n", i, f/norm)
}

fmt.Printf("\nAmplitude embedding encodes %d features into %d qubits.\n",
	len(features4D), 2)
fmt.Println("With 10 qubits, you could encode up to 1024 features.")

=== Amplitude Embedding ===
4D feature vector [1, 2, 3, 4] encoded into 2 qubits.
Expected amplitudes (normalized):
  |00> = 0.1826
  |01> = 0.3651
  |10> = 0.5477
  |11> = 0.7303

Amplitude embedding encodes 4 features into 2 qubits.
With 10 qubits, you could encode up to 1024 features.


q0 
 
 q1 
 
 
 Prep 
 Prep

## Variational Quantum Classifier (VQC)

A **Variational Quantum Classifier** is a hybrid classical-quantum model.
The quantum circuit has two parts:

1. **Feature map**: encodes the input data $\vec{x}$ into a quantum state.
2. **Variational ansatz**: a parameterized circuit $U(\vec{\theta})$ whose
   parameters $\vec{\theta}$ are optimized to minimize classification loss.

The full circuit is $U(\vec{\theta}) \cdot V(\vec{x}) |0\rangle$, and the
prediction is based on measuring qubit 0: $P(\text{class 1}) = 1 - |\langle 0|\psi\rangle|^2$.

Training uses a classical optimizer (here Nelder-Mead) to find the
parameters $\vec{\theta}^*$ that minimize the MSE loss over the training set.

```
Classical data  -->  Feature Map  -->  Ansatz(theta)  -->  Measure  -->  Prediction
     x                V(x)|0>          U(theta)           P(class)
                                         ^
                                         |
                              Classical optimizer updates theta
```

Let's create a simple 2D classification problem and train a VQC.

In [4]:
%%
// Simple classification: points above the y=x line (class 0) vs below (class 1)
trainX := [][]float64{
	{0.1, 0.9}, {0.2, 0.8}, {0.3, 0.7}, // class 0: above y=x
	{0.8, 0.2}, {0.9, 0.1}, {0.7, 0.3}, // class 1: below y=x
}
trainY := []int{0, 0, 0, 1, 1, 1}

fmt.Println("Training Data")
fmt.Println("=============")
fmt.Printf("%-12s  %s\n", "Point", "Label")
fmt.Println("--------------------")
for i, x := range trainX {
	side := "above y=x"
	if trainY[i] == 1 {
		side = "below y=x"
	}
	fmt.Printf("(%.1f, %.1f)     %d  (%s)\n", x[0], x[1], trainY[i], side)
}

// Show the ansatz structure
ans := ansatz.NewRealAmplitudes(2, 2, ansatz.Linear)
ansCirc, _ := ans.Circuit()
fmt.Printf("\nAnsatz: RealAmplitudes(2 qubits, 2 reps, Linear entanglement)\n")
fmt.Printf("Parameters: %d\n", ans.NumParams())
gonbui.DisplayHTML(draw.SVG(ansCirc))

Training Data
Point         Label
--------------------
(0.1, 0.9)     0  (above y=x)
(0.2, 0.8)     0  (above y=x)
(0.3, 0.7)     0  (above y=x)
(0.8, 0.2)     1  (below y=x)
(0.9, 0.1)     1  (below y=x)
(0.7, 0.3)     1  (below y=x)

Ansatz: RealAmplitudes(2 qubits, 2 reps, Linear entanglement)
Parameters: 6


q0 
 
 q1 
 
 RY 
 RY 
 
 
 
 RY 
 RY 
 
 
 
 RY 
 RY

In [5]:
%%
// Train the VQC
ctx := context.Background()

trainX := [][]float64{
	{0.1, 0.9}, {0.2, 0.8}, {0.3, 0.7},
	{0.8, 0.2}, {0.9, 0.1}, {0.7, 0.3},
}
trainY := []int{0, 0, 0, 1, 1, 1}

cfg := vqc.Config{
	NumQubits:  2,
	FeatureMap: vqc.AngleEmbedding(vqc.RotRY),
	Ansatz:     ansatz.NewRealAmplitudes(2, 2, ansatz.Linear),
	Optimizer:  &optim.NelderMead{},
	TrainX:     trainX,
	TrainY:     trainY,
}

result, err := vqc.Run(ctx, cfg)
if err != nil {
	fmt.Println("Error:", err)
} else {
	fmt.Printf("Training accuracy: %.2f%%\n", result.TrainAccuracy*100)
	fmt.Printf("Converged: %v, Iterations: %d\n", result.Converged, result.NumIters)
	fmt.Printf("Optimal parameters: [")
	for i, p := range result.OptimalParams {
		if i > 0 {
			fmt.Printf(", ")
		}
		fmt.Printf("%.4f", p)
	}
	fmt.Println("]")
}

Training accuracy: 100.00%
Converged: true, Iterations: 143
Optimal parameters: [-0.1501, -0.0547, 0.4158, -1.3227, 0.5294, -0.3601]


In [6]:
%%
// Make predictions on test data
ctx := context.Background()

trainX := [][]float64{
	{0.1, 0.9}, {0.2, 0.8}, {0.3, 0.7},
	{0.8, 0.2}, {0.9, 0.1}, {0.7, 0.3},
}
trainY := []int{0, 0, 0, 1, 1, 1}

cfg := vqc.Config{
	NumQubits:  2,
	FeatureMap: vqc.AngleEmbedding(vqc.RotRY),
	Ansatz:     ansatz.NewRealAmplitudes(2, 2, ansatz.Linear),
	Optimizer:  &optim.NelderMead{},
	TrainX:     trainX,
	TrainY:     trainY,
}

result, err := vqc.Run(ctx, cfg)
if err != nil {
	panic(err)
}

// Predict on unseen test points
testX := [][]float64{{0.1, 0.8}, {0.8, 0.1}, {0.5, 0.5}}
preds, err := vqc.Predict(cfg, result.OptimalParams, testX)
if err != nil {
	panic(err)
}

fmt.Println("Test Predictions")
fmt.Println("================")
for i, x := range testX {
	expected := "above y=x (class 0)"
	if x[0] > x[1] {
		expected = "below y=x (class 1)"
	}
	fmt.Printf("  (%.1f, %.1f) -> class %d  (expected: %s)\n", x[0], x[1], preds[i], expected)
}

fmt.Println("\nThe midpoint (0.5, 0.5) lies exactly on the decision boundary.")
fmt.Println("Its classification depends on the learned parameters.")

Test Predictions
  (0.1, 0.8) -> class 0  (expected: above y=x (class 0))
  (0.8, 0.1) -> class 1  (expected: below y=x (class 1))
  (0.5, 0.5) -> class 1  (expected: above y=x (class 0))

The midpoint (0.5, 0.5) lies exactly on the decision boundary.
Its classification depends on the learned parameters.


## Quantum Kernels

A **quantum kernel** measures the similarity between two data points in
quantum feature space. Given a feature map $V$, the kernel entry is:

$$K(\vec{x}_i, \vec{x}_j) = |\langle 0 | V^\dagger(\vec{x}_j) \cdot V(\vec{x}_i) | 0 \rangle|^2$$

This is the **fidelity** between the two feature states -- it equals 1
when the states are identical and approaches 0 when they are orthogonal.

The kernel matrix $K$ can be computed on a quantum computer and then used
with classical kernel methods (SVM, kernel ridge regression). This separates
the quantum advantage (computing kernels in exponential feature spaces) from
the classical training (optimizing a convex problem with the kernel matrix).

Key properties:
- $K(\vec{x}, \vec{x}) = 1$ always (self-similarity).
- $K$ is symmetric and positive semi-definite.
- Entangling feature maps (ZZ) create kernels that are classically hard
  to compute, which is where quantum advantage may arise.

In [7]:
%%
// Compute the quantum kernel matrix using the ZZ feature map
ctx := context.Background()

trainX := [][]float64{
	{0.1, 0.9}, {0.2, 0.8}, {0.3, 0.7},
	{0.8, 0.2}, {0.9, 0.1}, {0.7, 0.3},
}

kcfg := vqc.KernelConfig{
	NumQubits:  2,
	FeatureMap: vqc.ZZFeatureMap(2),
}

K, err := vqc.KernelMatrix(ctx, kcfg, trainX)
if err != nil {
	panic(err)
}

fmt.Println("Quantum Kernel Matrix (ZZ Feature Map, depth=2)")
fmt.Println("================================================")
fmt.Println("\nLabels: [0, 0, 0, 1, 1, 1]")
fmt.Println()

// Print header
fmt.Printf("%6s", "")
for i := range K {
	fmt.Printf("  x%-4d", i)
}
fmt.Println()

for i, row := range K {
	fmt.Printf("x%-4d", i)
	for _, v := range row {
		fmt.Printf("  %.3f", v)
	}
	fmt.Println()
}

fmt.Println("\nDiagonal entries are 1.000 (self-similarity).")
fmt.Println("Same-class pairs should have higher kernel values than cross-class pairs.")

Quantum Kernel Matrix (ZZ Feature Map, depth=2)

Labels: [0, 0, 0, 1, 1, 1]

        x0     x1     x2     x3     x4     x5   
x0     1.000  0.993  0.974  0.734  0.680  0.792
x1     0.993  1.000  0.993  0.788  0.734  0.843
x2     0.974  0.993  1.000  0.843  0.792  0.894
x3     0.734  0.788  0.843  1.000  0.993  0.993
x4     0.680  0.734  0.792  0.993  1.000  0.974
x5     0.792  0.843  0.894  0.993  0.974  1.000

Diagonal entries are 1.000 (self-similarity).
Same-class pairs should have higher kernel values than cross-class pairs.


In [8]:
%%
// Compute individual kernel entries to examine specific pair similarities
trainX := [][]float64{
	{0.1, 0.9}, {0.2, 0.8}, {0.3, 0.7},
	{0.8, 0.2}, {0.9, 0.1}, {0.7, 0.3},
}

kcfg := vqc.KernelConfig{
	NumQubits:  2,
	FeatureMap: vqc.ZZFeatureMap(2),
}

// Same class: x0 (class 0) vs x1 (class 0)
k01, _ := vqc.KernelEntry(kcfg, trainX[0], trainX[1])
fmt.Printf("K(x0, x1) = %.4f  (both class 0, nearby points)\n", k01)

// Cross class: x0 (class 0) vs x5 (class 1)
k05, _ := vqc.KernelEntry(kcfg, trainX[0], trainX[5])
fmt.Printf("K(x0, x5) = %.4f  (class 0 vs class 1, distant points)\n", k05)

// Same class: x3 (class 1) vs x4 (class 1)
k34, _ := vqc.KernelEntry(kcfg, trainX[3], trainX[4])
fmt.Printf("K(x3, x4) = %.4f  (both class 1, nearby points)\n", k34)

// Cross class: x2 (class 0) vs x3 (class 1)
k23, _ := vqc.KernelEntry(kcfg, trainX[2], trainX[3])
fmt.Printf("K(x2, x3) = %.4f  (class 0 vs class 1, distant points)\n", k23)

fmt.Println("\nA good feature map produces high kernel values for same-class pairs")
fmt.Println("and low values for cross-class pairs, enabling linear separation in")
fmt.Println("the quantum feature space.")

K(x0, x1) = 0.9932  (both class 0, nearby points)
K(x0, x5) = 0.7916  (class 0 vs class 1, distant points)
K(x3, x4) = 0.9932  (both class 1, nearby points)
K(x2, x3) = 0.8432  (class 0 vs class 1, distant points)

A good feature map produces high kernel values for same-class pairs
and low values for cross-class pairs, enabling linear separation in
the quantum feature space.


## Predict, Then Verify

**Question:** Will the VQC correctly classify the midpoint $(0.5, 0.5)$?

**Prediction:** The point $(0.5, 0.5)$ lies exactly on the $y = x$
decision boundary. Our training data has class 0 points above the line
and class 1 points below it. Since $(0.5, 0.5)$ is exactly on the
boundary, the classifier's prediction depends on the learned decision
surface. With a linear separation model, it could go either way.

For the clearly separated test points $(0.1, 0.8)$ and $(0.8, 0.1)$,
the VQC should classify them correctly as class 0 and class 1
respectively, since they are well within their class regions.

Let's verify.

In [9]:
%%
ctx := context.Background()

trainX := [][]float64{
	{0.1, 0.9}, {0.2, 0.8}, {0.3, 0.7},
	{0.8, 0.2}, {0.9, 0.1}, {0.7, 0.3},
}
trainY := []int{0, 0, 0, 1, 1, 1}

cfg := vqc.Config{
	NumQubits:  2,
	FeatureMap: vqc.AngleEmbedding(vqc.RotRY),
	Ansatz:     ansatz.NewRealAmplitudes(2, 2, ansatz.Linear),
	Optimizer:  &optim.NelderMead{},
	TrainX:     trainX,
	TrainY:     trainY,
}

result, err := vqc.Run(ctx, cfg)
if err != nil {
	panic(err)
}

fmt.Printf("Training accuracy: %.0f%%\n\n", result.TrainAccuracy*100)

// Test the specific predictions
testPoints := [][]float64{
	{0.1, 0.8}, // clearly above y=x -> class 0
	{0.8, 0.1}, // clearly below y=x -> class 1
	{0.5, 0.5}, // exactly on boundary -> ambiguous
}

preds, _ := vqc.Predict(cfg, result.OptimalParams, testPoints)

fmt.Println("Verification:")
fmt.Printf("  (0.1, 0.8) -> class %d  [predicted: class 0, above y=x]\n", preds[0])
fmt.Printf("  (0.8, 0.1) -> class %d  [predicted: class 1, below y=x]\n", preds[1])
fmt.Printf("  (0.5, 0.5) -> class %d  [on the boundary -- could go either way]\n", preds[2])

clearCorrect := (preds[0] == 0) && (preds[1] == 1)
fmt.Printf("\nClear points classified correctly: %v\n", clearCorrect)
fmt.Println("The boundary point (0.5, 0.5) classification depends on the learned parameters.")

Training accuracy: 100%

Verification:
  (0.1, 0.8) -> class 0  [predicted: class 0, above y=x]
  (0.8, 0.1) -> class 1  [predicted: class 1, below y=x]
  (0.5, 0.5) -> class 1  [on the boundary -- could go either way]

Clear points classified correctly: true
The boundary point (0.5, 0.5) classification depends on the learned parameters.


## Self-Check Questions

Before attempting the exercises, make sure you can answer these:

1. Why might a quantum kernel outperform a classical kernel on some datasets but not others?
2. What is the difference between angle embedding and amplitude embedding in terms of qubit efficiency?
3. Why does a quantum kernel matrix always have 1.0 on its diagonal?

---

## Exercises

### Exercise 1 -- Compare Feature Maps

Train the VQC with three different feature maps (angle embedding, Z feature
map, and ZZ feature map) on the same dataset. Compare their training
accuracy and convergence. Which feature map works best for this linearly
separable dataset?

In [10]:
%%
// Exercise 1: Compare Feature Maps
//
// Goal: Train the VQC with three different feature maps and compare
//       training accuracy and convergence speed.
//
// Feature maps to try:
//   - AngleEmbedding(RY)
//   - ZFeatureMap(depth=2)
//   - ZZFeatureMap(depth=2)
//
// TODO: Set up the training data
// TODO: Loop over feature maps, train a VQC with each, and print results
//
// Skeleton:

// ctx := context.Background()

// Step 1: Training data (above/below y=x line)
// trainX := [][]float64{
//     {0.1, 0.9}, {0.2, 0.8}, {0.3, 0.7}, // class 0
//     {0.8, 0.2}, {0.9, 0.1}, {0.7, 0.3}, // class 1
// }
// trainY := []int{0, 0, 0, 1, 1, 1}

// Step 2: Define the feature maps to compare
// type fmEntry struct {
//     name string
//     fm   vqc.FeatureMap
// }
// featureMaps := []fmEntry{
//     {"AngleEmbedding(RY)", vqc.AngleEmbedding(vqc.RotRY)},
//     {"ZFeatureMap(2)", vqc.ZFeatureMap(???)},
//     {"ZZFeatureMap(2)", vqc.ZZFeatureMap(???)},
// }

// Step 3: Train and compare
// fmt.Printf("%-22s  %10s  %10s  %10s\n", "Feature Map", "Accuracy", "Iters", "Converged")
// for _, entry := range featureMaps {
//     cfg := vqc.Config{
//         NumQubits:  2,
//         FeatureMap: ???,
//         Ansatz:     ansatz.NewRealAmplitudes(2, 2, ansatz.Linear),
//         Optimizer:  &optim.NelderMead{},
//         TrainX:     ???,
//         TrainY:     ???,
//     }
//     result, err := vqc.Run(ctx, cfg)
//     // Print: entry.name, result.TrainAccuracy, result.NumIters, result.Converged
// }

fmt.Println("TODO: Uncomment the code above, fill in the ???, and run!")

TODO: Uncomment the code above, fill in the ???, and run!


### Exercise 2 -- Effect of Ansatz Depth

The ansatz depth (number of repetitions) controls model expressivity.
More layers mean more parameters, which can fit more complex decision
boundaries but may also overfit or be harder to optimize.

Train the VQC with 1, 2, and 3 repetitions of the RealAmplitudes ansatz.
Compare the training accuracy and number of parameters.

In [11]:
%%
// Exercise 2: Effect of Ansatz Depth
//
// Goal: Train the VQC with 1, 2, and 3 repetitions of the RealAmplitudes
//       ansatz and compare training accuracy and parameter count.
//
// TODO: Set up training data
// TODO: Loop over repetition counts (1, 2, 3), train, and print results
//
// Skeleton:

// ctx := context.Background()

// Step 1: Training data
// trainX := [][]float64{
//     {0.1, 0.9}, {0.2, 0.8}, {0.3, 0.7},
//     {0.8, 0.2}, {0.9, 0.1}, {0.7, 0.3},
// }
// trainY := []int{0, 0, 0, 1, 1, 1}

// Step 2: Sweep ansatz depth
// fmt.Printf("%-8s  %8s  %10s  %10s  %10s\n",
//     "Reps", "Params", "Accuracy", "Iters", "Converged")
//
// for _, reps := range []int{1, 2, 3} {
//     ans := ansatz.NewRealAmplitudes(2, ???, ansatz.Linear)
//     cfg := vqc.Config{
//         NumQubits:  2,
//         FeatureMap: vqc.AngleEmbedding(vqc.RotRY),
//         Ansatz:     ???,
//         Optimizer:  ???,
//         TrainX:     ???,
//         TrainY:     ???,
//     }
//     result, err := vqc.Run(ctx, cfg)
//     // Print: reps, ans.NumParams(), result.TrainAccuracy, result.NumIters, result.Converged
// }

fmt.Println("TODO: Uncomment the code above, fill in the ???, and run!")

TODO: Uncomment the code above, fill in the ???, and run!


## Key Takeaways

1. **Quantum feature maps** encode classical data into quantum states.
   Angle embedding maps one feature per qubit (linear). Amplitude
   embedding encodes $2^n$ features in $n$ qubits (exponential
   compression). The ZZ feature map adds entangling gates that capture
   pairwise feature interactions.

2. **The Variational Quantum Classifier (VQC)** is a hybrid model:
   a quantum circuit (feature map + parameterized ansatz) produces
   predictions, and a classical optimizer (Nelder-Mead, Adam, etc.)
   tunes the ansatz parameters to minimize classification loss.

3. **Quantum kernels** measure similarity between data points in quantum
   feature space. The kernel matrix $K_{ij} = |\langle\phi(x_j)|\phi(x_i)\rangle|^2$
   can be computed on a quantum computer and fed to classical SVM or
   kernel ridge regression.

4. **Feature map choice matters.** Simpler maps (angle embedding) work
   for linearly separable data. More expressive maps (ZZ) may be needed
   for complex decision boundaries, but they also increase circuit depth
   and may be harder to train.

5. **Ansatz depth is a tradeoff.** More repetitions increase expressivity
   but also increase the number of parameters and the risk of barren
   plateaus (vanishing gradients in the cost landscape).

6. **Quantum advantage in ML is not automatic.** The potential advantage
   comes from quantum feature maps that create classically hard-to-compute
   kernels. For many practical datasets, classical models remain
   competitive or superior.

---

**Next up:** Notebook 15, where we will explore quantum error mitigation
and techniques for reducing the impact of noise on near-term quantum devices.